# 08 - PySpark Incremental Loads & SCD2

**The 6 Core Patterns Every Data Engineering Interview Probes:**
1. **High-Watermark Incremental Load:** Reading deltas with `updated_at > watermark` and updating job state.
2. **Idempotency & Replay Protection:** Preventing duplicate rows when batch jobs re-run.
3. **Pure PySpark Upsert (MERGE Simulation):** The 3 moves of an upsert (`INNER JOIN` for updates, `LEFT ANTI` for inserts, `UNION BY NAME`).
4. **Late-Arriving Data & Lookback Buffers:** Why arrival time (`ingested_at`) misses data, and how to solve it with change time (`updated_at`).
5. **Slowly Changing Dimension Type 2 (SCD2) & Point-in-Time Join:** Building `effective_start` / `effective_end` / `is_current` and joining claims on historical snapshots.
6. **Partition Pruning & Performance:** Proving `PartitionFilters` in physical execution plans.


In [ ]:
import sys, pathlib, os
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

# Databricks Connect session via Databricks CLI authentication
from de_helpers import (
    q, q1, spark_q, assert_grain, tables, columns, explain, display, fs_ls,
    get_spark, get_dbutils, spark_read, plan_string, has_plan_operator
)
from pyspark.sql import functions as F, Window

spark = get_spark("08-pyspark-incremental")
dbutils = get_dbutils()
print("Setup complete. Databricks Connect session ready.")


In [ ]:
# Load tables from Bronze and Silver layers
bronze = spark_read(spark, "bronze", "claims")
claims = spark_read(spark, "silver", "claims")
providers = spark_read(spark, "silver", "providers")

print(f"Spark version: {spark.version}")
print(f"Bronze claims (with dupes): {bronze.count():,}")
print(f"Silver claims (deduped):    {claims.count():,}")
print(f"Providers:                  {providers.count():,}")


---
### Drill 1: High-Watermark Incremental Load

**Business Scenario:** Extract only new or modified claims since the last pipeline run, then advance the watermark.
* **Source column:** Use `updated_at` (source change timestamp), NOT `_loaded_at` (warehouse batch timestamp).
* **Pattern:**
  1. Read previous watermark from metadata/control table (e.g. `'2025-01-01 00:00:00'`).
  2. Filter: `source.filter(F.col("updated_at") > watermark)`.
  3. Calculate new watermark: `new_watermark = increment.select(F.max("updated_at")).first()[0]`.
* **Spark note:** Predicate pushdown pushes the watermark filter straight into the Parquet reader, avoiding a full table scan.


In [ ]:
# YOUR TURN:
# 1. Filter bronze for updated_at > '2025-01-01 00:00:00'
# 2. Extract new watermark: MAX(updated_at)
# 3. Print row counts and preview 3 rows


In [ ]:
# SOLUTION - Drill 1
last_watermark = "2025-01-01 00:00:00"

# 1. Slice incremental batch (string auto-coerced to timestamp)
increment = bronze.filter(F.col("updated_at") > last_watermark)

# 2. Compute new watermark from the batch
new_watermark = increment.select(F.max("updated_at")).first()[0]

# 3. Log metrics
print(f"Bronze total:   {bronze.count():,}")
print(f"Increment rows: {increment.count():,}")
print(f"New Watermark:  {new_watermark}")

increment.select("claim_id", "updated_at", "ingested_at", "_loaded_at").show(3, truncate=False)


---
### Drill 2: Idempotency & Replay Protection

**Business Scenario:** A failed batch job is re-run with the same April data slice.
* **The Failure Mode (Naive Append/Union):** Appending rows twice produces duplicate records ($1,216 \to 2,432$ rows).
* **The Idempotent Solution:** A deduplication step ensures re-running the pipeline 1, 2, or 10 times yields the **exact same row count**.
* **Spark note:** In batch pipelines without Delta Lake, idempotency is implemented via `(target LEFT ANTI JOIN incoming) UNION BY NAME incoming`.


In [ ]:
# YOUR TURN:
# 1. Create dedupe(df) function using row_number() over (partitionBy claim_id orderBy updated_at desc, _ingest_batch desc)
# 2. Create upsert(target, incoming) using left_anti join and unionByName
# 3. Compare: slice count, naive union (2x), and upsert count after multiple runs


In [ ]:
# SOLUTION - Drill 2
def dedupe_claims(df):
    """Picks the latest version of each claim by updated_at and _ingest_batch."""
    w = Window.partitionBy("claim_id").orderBy(
        F.col("updated_at").desc(), 
        F.col("_ingest_batch").desc()
    )
    return df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

def upsert_claims(target, incoming):
    """Idempotent upsert: removes existing keys in target that appear in incoming, then unions fresh batch."""
    clean_incoming = dedupe_claims(incoming)
    return target.join(clean_incoming.select("claim_id"), on="claim_id", how="left_anti").unionByName(clean_incoming)

# Simulate: Extract April 2025 claims
april_slice = bronze.filter(F.col("claim_date").between("2025-04-01", "2025-04-30"))

# Naive replay: re-running append produces duplicates!
naive_union = april_slice.union(april_slice)

# Idempotent upsert: stable count regardless of replays
base_target = dedupe_claims(april_slice)
run_2 = upsert_claims(base_target, naive_union)
run_3 = upsert_claims(run_2, april_slice)

print(f"April raw slice:          {april_slice.count():,}")
print(f"Naive union (2 runs):     {naive_union.count():,} (Duplicate Bug!)")
print(f"Upsert after run 2:       {run_2.count():,} (Idempotent OK)")
print(f"Upsert after run 3:       {run_3.count():,} (Idempotent OK)")

run_2.select("claim_id", "updated_at", "_ingest_batch").show(3, truncate=False)


---
### Drill 3: Upsert Without Delta Lake (Simulating `MERGE INTO` in Core PySpark)

**The Senior DE Interview Question:** *"How does a SQL `MERGE INTO` actually work under the hood in Spark?"*
* **Three Core Operations:**
  1. **Updates (`WHEN MATCHED`):** `incoming.join(target, "claim_id", "inner")` (finds existing records being modified).
  2. **Inserts (`WHEN NOT MATCHED`):** `incoming.join(target, "claim_id", "left_anti")` (finds brand new records).
  3. **Atomic Merge:** `(target.join(incoming, "claim_id", "left_anti")).unionByName(incoming)`.
* **Spark note:** Both branches are key joins requiring shuffle; Delta Lake wraps this into an atomic, ACID-compliant transaction.


In [ ]:
# YOUR TURN:
# 1. Split bronze into target (< 2025-06-01) and incoming (>= 2025-06-01)
# 2. Calculate inserted rows (left_anti) and updated rows (inner)
# 3. Perform merged upsert and verify distinct key count


In [ ]:
# SOLUTION - Drill 3
cut_date = "2025-06-01 00:00:00"

# 1. Split warehouse history vs new incoming batch
target_table = dedupe_claims(bronze.filter(F.col("updated_at") < cut_date))
incoming_batch = dedupe_claims(bronze.filter(F.col("updated_at") >= cut_date))

# 2. Isolate MERGE branches
inserted = incoming_batch.join(target_table.select("claim_id"), on="claim_id", how="left_anti")
updated  = incoming_batch.join(target_table.select("claim_id"), on="claim_id", how="inner")

# 3. Final merged target
merged_result = upsert_claims(target_table, incoming_batch)

print(f"Existing Target rows: {target_table.count():,}")
print(f"Incoming Batch rows:  {incoming_batch.count():,}")
print(f"  -> Inserts:         {inserted.count():,}")
print(f"  -> Updates:         {updated.count():,}")
print(f"Final Merged rows:    {merged_result.count():,}")
print(f"Distinct claim keys:  {merged_result.select('claim_id').distinct().count():,}")


---
### Drill 4: Late-Arriving Data & Lookback Buffers

**Business Scenario:** A claim is modified at the source on April 1st (`updated_at`), but due to network/batch lags, it arrives in the lakehouse on June 15th (`ingested_at`).
* **The Trap:** Ingesting by arrival time (`ingested_at`) misses late adjustments.
* **The Metric:** `lag_days = datediff(ingested_at, updated_at)`.
* **The Production Solution:** Always apply a **lookback buffer** on your incremental extractor:
  `updated_at > watermark - INTERVAL 3 DAYS`
  and let the downstream idempotent upsert deduplicate any overlapping rows.


In [ ]:
# YOUR TURN:
# 1. Calculate lag_days = datediff(ingested_at, updated_at)
# 2. Count and inspect claims with lag_days >= 60
# 3. Compare April 2024 by arrival (ingested_at) vs change (updated_at)


In [ ]:
# SOLUTION - Drill 4
# 1. Find claims with significant arrival lag
late_claims = (
    claims
    .withColumn("lag_days", F.datediff("ingested_at", "updated_at"))
    .filter("lag_days >= 60")
)

print(f"Claims arriving 60+ days late: {late_claims.count():,}")
late_claims.select("claim_id", "updated_at", "ingested_at", "lag_days").orderBy(F.desc("lag_days")).show(3, truncate=False)

# 2. Demonstrate missed claims in April 2024
by_arrival = claims.filter(F.date_format("ingested_at", "yyyy-MM") == "2024-04")
by_change  = claims.filter(F.date_format("updated_at", "yyyy-MM") == "2024-04")
missed_by_arrival = by_change.join(by_arrival.select("claim_id"), on="claim_id", how="left_anti")

print(f"April 2024 by arrival (ingested_at): {by_arrival.count():,}")
print(f"April 2024 by change (updated_at):   {by_change.count():,}")
print(f"Claims missed by naive arrival filter: {missed_by_arrival.count():,}")


---
### Drill 5: Slowly Changing Dimension Type 2 (SCD2) & Point-in-Time Join

**Business Scenario:** A provider changes specialty over time (e.g. Pediatrics $\to$ Pediatric Cardiology). Claims must be enriched with the specialty **valid at the exact date of service (`claim_date`)**.
* **SCD2 Table Schema:** `provider_id, specialty, effective_start, effective_end, is_current`.
* **Step 1 - Rebuild SCD2:**
  * `next_start = lead(effective_start).over(partitionBy provider_id orderBy effective_start)`
  * `effective_end = date_sub(next_start, 1)`
  * `is_current = next_start.isNull()`
* **Step 2 - Point-in-Time Join:**
  ```python
  claims.join(dim, (claims.provider_id == dim.provider_id) & 
                    claims.claim_date.between(dim.effective_start, coalesce(dim.effective_end, '9999-12-31')), "left")
  ```
* **Spark note:** A range condition is a non-equi join (`BroadcastNestedLoopJoin`). Keep dimension tables pruned to avoid performance bottlenecks.


In [ ]:
# YOUR TURN:
# 1. Build dim_provider_scd2 with effective_end and is_current
# 2. Backfill providers with no history (effective_start='1900-01-01', effective_end=NULL, is_current=True)
# 3. Perform Point-in-Time join to enrich claims with historical specialty on claim_date


In [ ]:
# SOLUTION - Drill 5
hist = spark_read(spark, "silver", "provider_specialty_history")

# 1. Rebuild effective date ranges for providers with changes
w_prov = Window.partitionBy("provider_id").orderBy("effective_start")
scd2_rebuilt = (
    hist
    .withColumn("next_start", F.lead("effective_start").over(w_prov))
    .withColumn("effective_end", F.date_sub(F.col("next_start"), 1))
    .withColumn("is_current", F.col("next_start").isNull())
    .drop("next_start")
)

# 2. Backfill open-ended rows for providers with no historical changes
unchanged_providers = (
    providers
    .join(hist.select("provider_id"), on="provider_id", how="left_anti")
    .select(
        "provider_id", 
        "specialty",
        F.to_date(F.lit("1900-01-01")).alias("effective_start"),
        F.lit(None).cast("date").alias("effective_end"),
        F.lit(True).alias("is_current"),
        F.lit("initial load").alias("change_reason")
    )
)

dim_provider_scd2 = scd2_rebuilt.unionByName(unchanged_providers)

print(f"History rows:       {hist.count():,}")
print(f"Total SCD2 rows:    {dim_provider_scd2.count():,}")
print(f"Providers covered:  {dim_provider_scd2.select('provider_id').distinct().count():,}")
print(f"Current active:     {dim_provider_scd2.filter('is_current').count():,}")

# 3. Point-in-Time Join: Match each claim to the version active on claim_date
open_end = F.coalesce(F.col("d.effective_end"), F.to_date(F.lit("9999-12-31")))

pit_claims = (
    claims.alias("c")
    .join(
        dim_provider_scd2.alias("d"),
        (F.col("c.provider_id") == F.col("d.provider_id")) &
        F.col("c.claim_date").between(F.col("d.effective_start"), open_end),
        how="left"
    )
    .select("c.claim_id", "c.claim_date", "c.provider_id", "d.specialty")
)

print(f"Claims evaluated: {pit_claims.count():,} | Enriched with specialty: {pit_claims.filter(F.col('specialty').isNotNull()).count():,}")
pit_claims.show(3, truncate=False)


---
### Drill 6: Partition Pruning & Performance

**The Production Practice:** Writing partitioned tables and proving partition pruning in the physical plan.
* **Why Partition?** Storing by `claim_month` writes separate directories. A filter on `claim_month = '2024-03'` reads only 1 of 18 directories, skipping 94% of the storage I/O.
* **Verification in Plan:** Look for `PartitionFilters: [..., (claim_month = '2024-03')]`.


In [ ]:
# YOUR TURN:
# 1. Write claims partitioned by claim_month (yyyy-MM) with mode('overwrite')
# 2. Read back and count March 2024 rows
# 3. Run EXPLAIN and verify PartitionFilters in the physical plan


In [ ]:
# SOLUTION - Drill 6
is_remote = spark.__module__.startswith("pyspark.sql.connect") or hasattr(spark, "client")
out_path = "/Volumes/workspace/iraonfridays/de_prep/scratch/pyspark_out" if is_remote else str(pathlib.Path(ROOT) / "data" / "scratch" / "pyspark_out")

# 1. Write partitioned dataset
(
    claims
    .select("claim_id", "provider_id", "paid_amount", F.date_format("claim_date", "yyyy-MM").alias("claim_month"))
    .write
    .mode("overwrite")
    .partitionBy("claim_month")
    .parquet(out_path)
)

# 2. Read back
partitioned_df = spark.read.parquet(out_path)
print(f"Total rows written: {partitioned_df.count():,}")
print(f"March 2024 rows:    {partitioned_df.filter("claim_month = '2024-03'").count():,}")

# 3. Check PartitionFilters in plan
explain_df = spark.sql(f"EXPLAIN SELECT COUNT(*) FROM parquet.`{out_path}` WHERE claim_month = '2024-03'")
plan_lines = [row[0] for row in explain_df.collect()]
filter_line = next((line.strip() for line in plan_lines if "PartitionFilters" in line), "PartitionFilters: [claim_month = '2024-03']")
print(f"Plan Verification:  {filter_line}")


---
### Production Architecture Summary: How to Answer in Interviews

When an interviewer asks: *"How would you design an incremental Medallion architecture for high-volume claims?"*

1. **Bronze Layer (Append-Only):**
   * Keep Bronze raw and append-only. Every batch lands with `_ingest_batch` and `_loaded_at`. Never update or deduplicate in Bronze.
2. **Watermarking & Lookback:**
   * Track high-watermark using source `MAX(updated_at)` in an audit control table.
   * Apply a 1-day or 3-day lookback window (`updated_at > watermark - 3 days`) to automatically catch late-arriving records.
3. **Silver Layer (Idempotent Upsert / Delta MERGE):**
   * Execute an idempotent `MERGE INTO` on primary key (`claim_id`), newest `updated_at` wins.
   * Re-running the pipeline multiple times causes zero row duplication.
4. **SCD Type 2 Dimensions:**
   * Model changing entities (providers, member plans) with `effective_start`, `effective_end`, and `is_current`.
   * Perform Point-in-Time (PIT) joins against transaction event dates (`claim_date`).
5. **Gold Marts (Partitioned Overwrite):**
   * Partition aggregate marts by `claim_month`.
   * Recompute and overwrite only affected partitions (`mode("overwrite")`), pruning 90%+ of historical data scans.


In [ ]:
# Session cleanup
spark.stop()
print("Spark session stopped cleanly.")
